# 1. Research Question

## Research Question

Can we guess a song's musical genre just from its lyrics?

A country ballad and a hip-hop track feel completely different on the page — but is that intuition backed by measurable linguistic patterns? We frame this as a **multi-class text classification** problem: given only raw lyrics, predict whether a song is Rock, Pop, Hip-Hop, Metal, Country, Jazz, Electronic, Folk, R&B, or Indie.

- **Task:** Multi-class text classification.
- **Input:** Song lyrics (raw text).
- **Target variable:** Musical genre (categorical label).
- **Hypothesis:** Each genre leaves a distinct linguistic fingerprint — in vocabulary, syntax, and rhetoric — strong enough to beat a majority-class baseline.

**Expected observations:** Hip-Hop stands out through slang and self-reference; Country through rural imagery; Metal through aggressive vocabulary. Syntactically, Hip-Hop favors short lines while Folk leans narrative. Models that capture context and word order should outperform simpler bag-of-words approaches.

---

## Required Reflection

### 1. Why is this question non-trivial?

At first glance, guessing genre from lyrics sounds straightforward — but it really isn't, for a few reasons:

- **Genre boundaries are blurry.** A song tagged as Pop-Rock on Spotify doesn't neatly fit either category. Labels are often assigned by platforms or marketing, not by any linguistic criterion.
- **We're throwing away most of the signal.** Without melody, tempo, or instrumentation, the classifier only gets the text. That's a fraction of what actually defines a genre.
- **Huge variation within genres.** A quiet acoustic Rock ballad and an arena Rock anthem share a label but almost nothing else in their lyrics.
- **Class imbalance.** Rock dominates most lyrics datasets, which means the model could learn to just predict "Rock" and still look decent on accuracy.

### 2. What linguistic phenomenon does it involve?

At its core, this project is about **register variation conditioned by genre** — the idea that the social and artistic context of a song shapes how language is used. More concretely, we're looking at:

- **Lexical semantics:** Do genres prefer different words and word distributions?
- **Morpho-syntactic patterns:** Are there measurable differences in POS-tag usage, sentence length, or syntactic complexity across genres?
- **Pragmatics and rhetoric:** Do genres differ in their use of repetition, imperatives, interjections, or figurative language?
- **Sociolinguistic register:** How much do formality, slang density, and code-switching vary from one genre to another?

### 3. What would falsify our hypothesis?

We'd have to admit we were wrong if:

- All models — including the most expressive ones — perform around the majority-class baseline, which would mean lyrics simply don't carry enough genre signal.
- Our linguistic analyses show no statistically significant vocabulary, syntactic, or rhetorical differences across genres.
- Misclassification errors are scattered randomly instead of clustering around genre pairs that are linguistically close (e.g., Rock/Metal, Folk/Country), which would suggest the models aren't really learning meaningful language patterns.

# 2. Data & Preprocessing

## Loading the dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("akyshnik/lyrics")

print("Path to dataset files:", path)

Path to dataset files: /home/alberto/.cache/kagglehub/datasets/akyshnik/lyrics/versions/1


In [2]:
import os

print(os.listdir(path))

['kaggle_clean.csv']


In [3]:
import pandas as pd

file_path = os.path.join(path, "kaggle_clean.csv")
df = pd.read_csv(file_path)

## 2.1 Dataset Selection

### Checking the values

In [4]:
df["genre"].unique()

array(['Pop', 'Hip-Hop', 'Not Available', 'Rock', 'Metal', 'Other',
       'Country', 'Jazz', 'Electronic', 'Folk', 'R&B', 'Indie'],
      dtype=object)

In [5]:
df.info()
df.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266557 entries, 0 to 266556
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  266557 non-null  int64 
 1   index       266557 non-null  int64 
 2   song        266556 non-null  object
 3   year        266557 non-null  int64 
 4   artist      266557 non-null  object
 5   genre       266557 non-null  object
 6   lyrics      266535 non-null  object
 7   wnum        266557 non-null  int64 
dtypes: int64(4), object(4)
memory usage: 16.3+ MB


,Unnamed: 0,index,song,year,artist,genre,lyrics,wnum
93463,93463,128369,rise-n-shine,2006,extreme,Rock,dawn wakes the silence\nof a fainted lullaby\n...,237
73270,73270,101654,oeiiaiu,2006,aao,Rock,\n ! \n \n \n \n \n\n!\n !\n !\n !\n\n\n \n ...,64
185833,185833,252699,o-dear-minny-what-shall-i-do,2006,burns-robert,Not Available,o dear minny what shall i do?\nrobert burns\nc...,94
82555,82555,114794,joey,2006,concrete-blonde,Rock,joey baby don't get crazy\ndetours fencesi ge...,136
78710,78710,109256,mercury,2007,brave-saint-saturn,Rock,\nand in other news the crew of the uss gloria...,70
207476,207476,282089,ignorance-is-bliss,1999,alan-parsons,Rock,i find this paradise and rest beside a river\n...,201
217627,217627,296261,stranger,2006,gorky-park,Rock,stranger next to me have i seen your face\nstr...,219
32606,32606,45272,down-to-the-wateline,2007,dire-straits,Rock,sweet surrender on the quayside\nyou remember ...,146
110043,110043,150398,college,2010,animal-collective,Rock,ok\ncan we do it? can we do it?\nuhhhhhhhhh uh...,21
193022,193022,262483,tonight,2007,crystal-lewis,Pop,copyright barry hinesley bill baumgart tim hei...,123


### Modify useless values

In [6]:
df.loc[df["wnum"] == 1, "lyrics"].value_counts()

lyrics
instrumental     3645
instru             51
instumental        10
intro              10
intrumental         9
                 ... 
orchestra           1
finntroll           1
pow!                1
instermentual       1
bonus               1
Name: count, Length: 73, dtype: int64

In [7]:
df.loc[df["wnum"] == 1, "lyrics"] = "instrumental"
df.loc[df["wnum"] == 1, "lyrics"].value_counts()

lyrics
instrumental    3842
Name: count, dtype: int64

### Eliminate Non Available values

We do this because there is enough values to being able to eliminate every row that has incomplete information.

In [8]:
df_clean = df[~df["genre"].isin(["Not Available"])]
df_clean = df_clean[~df_clean["year"].isin([112, 702, 67])]

In [9]:
print(len(df), len(df_clean))

266557 242610


### Check impact of instrumental songs

In [10]:
print("Percentage of instrumental songs for each genre:\n")

inst = df_clean["genre"][df_clean["lyrics"] == "instrumental"].value_counts()
no_inst = df_clean["genre"][df_clean["lyrics"] != "instrumental"].value_counts()
genres = inst.keys()

for i in range(len(inst)):
    print(genres[i]+": ", round((inst.iloc[i]/no_inst.iloc[i])*100, 2), "\b%")

Percentage of instrumental songs for each genre:

Rock:  1.31%
Metal:  1.9%
Electronic:  1.4%
Pop:  0.88%
Jazz:  1.01%
Folk:  1.73%
Hip-Hop:  1.12%
Country:  1.23%
Indie:  1.33%
R&B:  0.32%
Other:  0.28%


### Dataset Description

Source: 
Domain: 
Label distribution: 
Potential biases: 
Ethical considerations: 